In [10]:
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
import sys

sys.path.append("..")

In [28]:
import importlib
from utils import neo4j_driver, num_tokens_from_string, chunk_text, chat, embed
import ch07_tools
importlib.reload(ch07_tools)

import json
import requests

from tqdm import tqdm
from typing import List, Dict

In [13]:
text = ""

with open("../data/pg1727.txt", "r") as f:
    text = f.read()

print(text[:500])

The Project Gutenberg eBook of The Odyssey
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

Title: 


In [14]:
def chunk_into_books(text: str) -> List[str]:
    return (
        text.split("PREFACE TO FIRST EDITION")[2]
        .split("FOOTNOTES")[0]
        .strip()
        .split("\nBOOK")[1:]
    )

books = chunk_into_books(text)

In [15]:
token_count = [num_tokens_from_string(el) for el in books]
print(
    f"""There are {len(token_count)} books with token sizes:
    - avg {sum(token_count) / len(token_count)}
    - min {min(token_count)}
    - max {max(token_count)}
"""
)

There are 24 books with token sizes:
    - avg 6486.791666666667
    - min 4444
    - max 10709



In [16]:
chunked_books = [chunk_text(book, 1000, 40) for book in books]

In [17]:
ENTITY_TYPES = [
    "PERSON",
    "ORGANIZATION",
    "LOCATION",
    "GOD",
    "EVENT",
    "CREATURE",
    "WEAPON_OR_TOOL",
]
def extract_entities(text: str) -> List[Dict]:
    # Construct prompt
    messages = [
        {"role": "user", "content": ch07_tools.create_extraction_prompt(ENTITY_TYPES, text)},
    ]
    # Make the LLM call
    output = chat(messages, model = "gpt-5.1")
    # Construct JSON from output
    return ch07_tools.parse_extraction_output(output)

In [18]:
number_of_books = 1
for book_i, book in enumerate(
    tqdm(chunked_books[:number_of_books], desc="Processing Books")
):
    for chunk_i, chunk in enumerate(tqdm(book, desc=f"Book {book_i}", leave=False)):
        nodes, relationships = extract_entities(chunk)
        neo4j_driver.execute_query(
            ch07_tools.import_nodes_query,
            data=nodes,
            book_id=book_i,
            text=chunk,
            chunk_id=chunk_i,
        )
        neo4j_driver.execute_query(
            ch07_tools.import_relationships_query, data=relationships
        )

Processing Books: 100%|██████████| 1/1 [15:26<00:00, 926.07s/it]


In [19]:
data, _, _ = neo4j_driver.execute_query(
    """MATCH (:`__Entity__`)
    RETURN 'entity' AS type, count(*) AS count
    UNION
    MATCH ()-[:RELATIONSHIP]->()
    RETURN 'relationship' AS type, count(*) AS count
    """
)
print([el.data() for el in data])

[{'type': 'entity', 'count': 179}, {'type': 'relationship', 'count': 559}]


In [20]:
data, _, _ = neo4j_driver.execute_query(
    """MATCH (n:PERSON)
WHERE n.name = "ORESTES"
RETURN n.description AS description"""
)
print([el.data()['description'] for el in data])

[['Orestes is the son of Agamemnon, who avenges his father’s murder by killing Aegisthus.', 'Orestes is an individual fated to take revenge on Aegisthus when he grew up and returned home.', "Orestes is a famed figure known for killing Aegisthus, who murdered his father. Orestes is celebrated in song and regarded as an exemplar of avenging one's family.", 'Orestes is the son of Agamemnon who kills Aegisthus in retribution for the murder of his father', "Orestes is a mortal son who, when grown, takes revenge on Aegisthus for his father's murder upon returning home.", 'Orestes is cited as a heroic figure whose fame comes from avenging his father by killing his father’s murderer, Aegisthus. His example is used to inspire Telemachus to take decisive action against the suitors.']]


In [21]:
data, _, _ = neo4j_driver.execute_query(
    """MATCH (n:__Entity__)-[:RELATIONSHIP]-(m:__Entity__)
WITH n,m, count(*) AS countOfRels
ORDER BY countOfRels DESC LIMIT 1
MATCH (n)-[r:RELATIONSHIP]-(m)
RETURN n.name AS source, m.name AS target, countOfRels, collect(r.description) AS descriptions
"""
)
print([el.data() for el in data])

[{'source': 'TELEMACHUS', 'target': 'ULYSSES', 'countOfRels': 14, 'descriptions': ['Telemachus believes his father Ulysses is dead and refers to rumors and prophecies about his return', 'Telemachus is the son of Ulysses and compares his hoped-for role as chief in Ithaca to that of his father', 'Telemachus identifies Ulysses as the man he is told is his father and is said to resemble him in head and eyes', 'Telemachus is the son of Ulysses and is tasked with seeking news of his father’s return', "Telemachus is the son of Ulysses and is concerned with his father's fate", 'Telemachus is the son of Ulysses, whose absence prompts the dispute over leadership', 'Ulysses is the father of Telemachus, whose presumed death shapes Telemachus’s situation', 'Ulysses is the father of Telemachus, who now asserts authority in his father’s house', 'Ulysses is the reputed father of Telemachus, who is described as wonderfully like him and as the continuation of his race', 'Telemachus is Ulysses’ son, impl

In [22]:
candidates_to_summarize, _, _ = neo4j_driver.execute_query(
    """MATCH (e:__Entity__) WHERE size(e.description) > 1 
    RETURN e.name AS entity_name, e.description AS description_list"""
)
summaries = []
for candidate in tqdm(candidates_to_summarize, desc="Summarizing entities"):
    messages = [
        {
            "role": "user",
            "content": ch07_tools.get_summarize_prompt(
                candidate["entity_name"], candidate["description_list"]
            ),
        },
    ]
    summary = chat(messages, model="gpt-5.4")
    summaries.append({"entity": candidate["entity_name"], "summary": summary})

ch07_tools.import_entity_summary(neo4j_driver, summaries)

Summarizing entities: 100%|██████████| 93/93 [03:58<00:00,  2.57s/it]


In [23]:
summary, _, _ = neo4j_driver.execute_query(
    """MATCH (n:PERSON)
WHERE n.name = "ORESTES"
RETURN n.summary AS summary""")
print(summary[0]['summary'])

Orestes is the son of Agamemnon and a mortal heroic figure famed for avenging his father’s murder. After growing up and returning home, he fulfills his fate by killing Aegisthus, the man who murdered Agamemnon. Orestes is celebrated in song and remembered as an exemplar of familial vengeance and decisive action, and his story is invoked as an inspiring model for Telemachus to act boldly against the suitors.


In [24]:
rels_to_summarize, _, _ = neo4j_driver.execute_query(
    """MATCH (s:__Entity__)-[r:RELATIONSHIP]-(t:__Entity__)
    WHERE id(s) < id(t)
    WITH s.name AS source, t.name AS target, 
           collect(r.description) AS description_list,
           count(*) AS count
    WHERE count > 1
    RETURN source, target, description_list"""
)
rel_summaries = []
for candidate in tqdm(rels_to_summarize, desc="Summarizing relationships"):
    entity_name = f"{candidate['source']} relationship to {candidate['target']}"
    messages = [
        {
            "role": "user",
            "content": ch07_tools.get_summarize_prompt(
                entity_name, candidate["description_list"]
            ),
        },
    ]
    summary = chat(messages, model="gpt-5.4-mini")
    rel_summaries.append({"source": candidate["source"], "target": candidate["target"], "summary": summary})

ch07_tools.import_rels_summary(neo4j_driver, rel_summaries)

Summarizing relationships: 100%|██████████| 127/127 [02:08<00:00,  1.01s/it]


In [25]:
data, _, _ = neo4j_driver.execute_query(
    """MATCH (n:__Entity__)-[r:SUMMARIZED_RELATIONSHIP]-(m:__Entity__)
WHERE n.name = 'TELEMACHUS' AND m.name = 'MINERVA'
RETURN r.summary AS description
"""
)
print(data[0]["description"])

Minerva has a close, guiding relationship with Telemachus during her visit to Ithaca. Telemachus is the first to notice Minerva and goes to greet her, welcoming and honoring her as a guest, though he initially believes her to be Mentes. During the banquet, he speaks quietly and confides his thoughts to Minerva so that no one else can hear. Minerva in turn speaks directly to him, questions him about the feasting, reassures him about his lineage, and praises him as Penelope’s fine son. She encourages him, gives him courage, and offers advice about his planned voyage. Her guidance leaves a strong impression on Telemachus, who thinks all night about the counsel Minerva has given him and is made to reflect more deeply than ever on his father.


In [26]:
community_distribution = ch07_tools.calculate_communities(neo4j_driver)
print(f"There are {community_distribution['communityCount']} communities with distribution: {community_distribution['communityDistribution']}")

There are 10 communities with distribution: {'p1': 1, 'max': 60, 'p5': 1, 'p90': 41, 'p50': 9, 'p95': 60, 'p10': 1, 'p75': 30, 'p99': 60, 'p25': 4, 'min': 1, 'mean': 17.9, 'p999': 60}


In [29]:
community_info, _, _ = neo4j_driver.execute_query(ch07_tools.community_info_query)

communities = []
for community in tqdm(community_info, desc="Summarizing communities"):
    messages = [
        {
            "role": "user",
            "content": ch07_tools.get_summarize_community_prompt(
                community["nodes"], community["rels"]
            ),
        },
    ]
    summary = chat(messages, model="gpt-5.4-mini")
    communities.append(
        {
            "community": json.loads(ch07_tools.extract_json(summary)),
            "communityId": community["communityId"],
            "nodes": [el["id"] for el in community["nodes"]],
        }
    )

neo4j_driver.execute_query(ch07_tools.import_community_query, data=communities)

Summarizing communities: 100%|██████████| 9/9 [02:01<00:00, 13.51s/it]


EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x7e915fa5b890>, keys=[])

In [30]:
data, _, _ = neo4j_driver.execute_query(
    """MATCH (c:__Community__)
WITH c, count {(c)<-[:IN_COMMUNITY]-()} AS size
ORDER BY size DESC LIMIT 1
RETURN c.title AS title, c.summary AS summary
"""
)
print(data[0]["title"])
print(data[0]["summary"])

Ithaca: Odysseus, Telemachus, Penelope, and the Suitors
This community centers on Ithaca and the House of Ulysses, where Odysseus is the absent rightful ruler and Telemachus, Penelope, and a large group of suitors are locked in a struggle over household authority and inheritance. Telemachus is coming of age under pressure from the suitors, who occupy the estate, consume its wealth, and seek Penelope’s hand, while Penelope remains loyal but grief-stricken and Phemius provides music at the feasts. The situation is further complicated by notable suitors such as Antinous and Eurymachus, the involvement of servants and banqueting arrangements, and Telemachus’s plans to confront the disorder through action, counsel from the goddess, and a proposed assembly of the Achaeans.


In [31]:
def global_retriever(query: str, rating_threshold: float = 5) -> str:
    community_data, _, _ = neo4j_driver.execute_query(
        """
    MATCH (c:__Community__)
    WHERE c.rating >= $rating
    RETURN c.summary AS summary
    """,
        rating=rating_threshold,
    )
    print(f"Got {len(community_data)} community summaries")
    intermediate_results = []
    for community in tqdm(community_data, desc="Processing communities"):
        intermediate_messages = [
            {
                "role": "system",
                "content": ch07_tools.get_map_system_prompt(community["summary"]),
            },
            {
                "role": "user",
                "content": query,
            },
        ]
        intermediate_response = chat(intermediate_messages, model="gpt-5.4-mini")
        intermediate_results.append(intermediate_response)

    final_messages = [
        {
            "role": "system",
            "content": ch07_tools.get_reduce_system_prompt(intermediate_results),
        },
        {"role": "user", "content": query},
    ]
    summary = chat(final_messages, model="gpt-5.4-mini")
    return summary

In [32]:
print(global_retriever("What is this story about?"))

Got 7 community summaries


Processing communities: 100%|██████████| 7/7 [01:10<00:00, 10.09s/it]


The story is principally about Odysseus (Ulysses) and his arduous journey back to Ithaca following the Trojan War, a journey fraught with trials both terrestrial and divine. The narrative unfolds over two primary loci: Ithaca, where Odysseus's household faces an internal crisis, and the broader mythological realm, where gods and mythical elements influence the human saga. 

### Ithaca's Crisis

At the heart of the story's dramatic tension is the situation in Ithaca, where Telemachus (Odysseus's son) and Penelope (his wife) are grappling with a cohort of opportunistic suitors. These suitors have overrun Odysseus's home, consuming its resources and seeking Penelope’s hand in marriage, thereby threatening the household's legacy in the absence of its rightful head [Data: Reports]. Telemachus, coming of age amidst this chaos, is notably growing in authority as he attempts to maintain order and plan for action against the suitors [Data: Entities (TELEMACHUS, SUITORS, HOUSE OF ULYSSES)].

Min

In [35]:
entities, _, _ = neo4j_driver.execute_query(
    """
MATCH (e:__Entity__)
RETURN e.summary AS summary, e.name AS name
"""
)
data = [{"name": el["name"], "embedding": embed(el["summary"] or el["name"]).tolist()} for el in entities]
neo4j_driver.execute_query(
    """
UNWIND $data AS row
MATCH (e:__Entity__ {name: row.name})
CALL db.create.setNodeVectorProperty(e, 'embedding', row.embedding)
""",
    data=data,
)

neo4j_driver.execute_query(
    """
CREATE VECTOR INDEX entities IF NOT EXISTS
FOR (n:__Entity__)
ON (n.embedding)
""",
    data=data,
)

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x7e90d98c2e10>, keys=[])

In [36]:
local_search_query = """
CALL db.index.vector.queryNodes('entities', $k, $embedding)
YIELD node, score
WITH collect(node) as nodes
WITH collect {
    UNWIND nodes as n
    MATCH (n)<-[:HAS_ENTITY]->(c:__Chunk__)
    WITH c, count(distinct n) as freq
    RETURN c.text AS chunkText
    ORDER BY freq DESC
    LIMIT $topChunks
} AS text_mapping,
collect {
    UNWIND nodes as n
    MATCH (n)-[:IN_COMMUNITY]->(c:__Community__)
    WITH c, c.rank as rank, c.weight AS weight
    RETURN c.summary 
    ORDER BY rank, weight DESC
    LIMIT $topCommunities
} AS report_mapping,
collect {
    UNWIND nodes as n
    MATCH (n)-[r:SUMMARIZED_RELATIONSHIP]-(m) 
    WHERE m IN nodes
    RETURN r.summary AS descriptionText
    ORDER BY r.rank, r.weight DESC 
    LIMIT $topInsideRels
} as insideRels,
collect {
    UNWIND nodes as n
    RETURN n.summary AS descriptionText
} as entities
RETURN {Chunks: text_mapping, Reports: report_mapping, 
       Relationships: insideRels, 
       Entities: entities} AS text
"""

In [39]:
k_entities = 5

topChunks = 3
topCommunities = 3
topInsideRels = 3


def local_search(query: str) -> str:
    context, _, _ = neo4j_driver.execute_query(
        local_search_query,
        embedding=embed(query).tolist(),
        topChunks=topChunks,
        topCommunities=topCommunities,
        topInsideRels=topInsideRels,
        k=k_entities,
    )
    context_str = str(context[0]["text"])
    local_messages = [
        {
            "role": "system",
            "content": ch07_tools.get_local_system_prompt(context_str),
        },
        {
            "role": "user",
            "content": query,
        },
    ]
    final_answer = chat(local_messages, model="gpt-4o")
    return final_answer

In [40]:
print(local_search("Who is Ulysses?"))

Ulysses, also known as Odysseus, is a legendary Greek hero known for his intelligence, cunning, and piety. He is the rightful king or chief of Ithaca, where he is recognized as a central figure both in his absence and presence [Data: Entities (None)]. Ulysses gained fame for his crucial role in the Trojan War, where his strategic acumen and leadership skills were instrumental in the fall of Troy. Celebrated as the hero who helped sack the city, his narratives are deeply interwoven with the mythic tales of ancient Greece [Data: Entities (None)].

His journey following the Trojan War is notable for its series of harrowing adventures and hardships. Ulysses endured numerous challenges at sea, striving to return home and ensure the safety of his crew. Unfortunately, he was unable to save all his men. Throughout these wanderings, he visited many cities and nations, characterizing his journey as one of exploration and survival [Data: Entities (None)]. Despite the odds, the deities have decree